# VisionBridge A-Z base model training

Run these cells from top to bottom.

This notebook is deliberately isolated from Colab's preinstalled Python packages. It creates a small virtual environment, installs the exact MediaPipe version used by VisionBridge, downloads the real RealSign Git-LFS archive, extracts the 126D hand representation, uses a stratified 80/20 train-validation split, trains for up to 500 epochs, and saves the best checkpoint.

The original RealSign testing split remains untouched.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

ROOT = Path("/content")
REPO = ROOT / "VisionBridge"
VENV = ROOT / "visionbridge_train_env"
PYTHON = VENV / "bin" / "python"
PIP = VENV / "bin" / "pip"

for path in (REPO, VENV):
    if path.exists():
        shutil.rmtree(path)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/BharathWaj-K-R/VisionBridge.git",
        str(REPO),
    ],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "venv", "--system-site-packages", str(VENV)],
    check=True,
)

subprocess.run(
    [
        str(PIP),
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "mediapipe==0.10.35",
    ],
    check=True,
)

env = dict(**__import__("os").environ)
env["PYTHONPATH"] = str(REPO / "backend")

smoke = r'''
import sys
import torch
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("MediaPipe:", mp.__version__)

assert mp.__version__ == "0.10.35"

print("MediaPipe Tasks API:", vision.HandLandmarker.__name__)
print("Training environment: PASS")
'''

subprocess.run([str(PYTHON), "-c", smoke], check=True, env=env)

In [ ]:
from pathlib import Path
import urllib.request

HAND_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
)
HAND_MODEL_PATH = Path("/content/hand_landmarker.task")

if HAND_MODEL_PATH.exists():
    HAND_MODEL_PATH.unlink()

print("Downloading MediaPipe Hand Landmarker model...")
urllib.request.urlretrieve(HAND_MODEL_URL, HAND_MODEL_PATH)

size = HAND_MODEL_PATH.stat().st_size
if size < 1_000_000:
    raise RuntimeError(f"Hand model download is unexpectedly small: {size} bytes")

print("Hand model bytes:", size)
print("Hand model download: PASS")

In [ ]:
import os
import subprocess
from pathlib import Path

PYTHON = Path("/content/visionbridge_train_env/bin/python")
MODEL = Path("/content/hand_landmarker.task")
ENV = os.environ.copy()

smoke = r'''
import sys
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_path = sys.argv[1]
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2,
)
detector = vision.HandLandmarker.create_from_options(options)
detector.close()
print("Hand Landmarker initialization: PASS")
'''

subprocess.run([str(PYTHON), "-c", smoke, str(MODEL)], check=True, env=ENV)

In [ ]:
from pathlib import Path
import shutil
import subprocess
import urllib.request
import zipfile

DATASET_URL = (
    "https://media.githubusercontent.com/media/RealSign62/"
    "RealSign-Indian-Sign-Language-Dataset/main/Dataset.zip"
)
ZIP_PATH = Path("/content/RealSign.zip")
DATASET_DIR = Path("/content/RealSign")

for path in (ZIP_PATH, DATASET_DIR):
    if path.is_file():
        path.unlink()
    elif path.is_dir():
        shutil.rmtree(path)

print("Downloading RealSign dataset...")
subprocess.run(
    [
        "curl",
        "-L",
        "--fail",
        "--retry",
        "5",
        "--retry-all-errors",
        "--output",
        str(ZIP_PATH),
        DATASET_URL,
    ],
    check=True,
)

if not zipfile.is_zipfile(ZIP_PATH):
    first_bytes = ZIP_PATH.read_text(errors="replace")[:200]
    raise RuntimeError(
        "RealSign download is not a valid ZIP archive. "
        "The Git LFS media request returned unexpected content: "
        + repr(first_bytes)
    )

print("RealSign archive bytes:", ZIP_PATH.stat().st_size)

with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall(DATASET_DIR)

print("RealSign extraction: PASS")

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/VisionBridge")
PYTHON = Path("/content/visionbridge_train_env/bin/python")
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "backend")

command = [
    str(PYTHON),
    str(REPO / "backend/scripts/prepare_letter_dataset.py"),
    "--input-root",
    "/content/RealSign",
    "--output-dir",
    "/content/visionbridge_letter_data",
    "--validation-ratio",
    "0.20",
    "--seed",
    "42",
    "--hand-model-path",
    "/content/hand_landmarker.task",
]

subprocess.run(command, check=True, env=ENV)
print("Landmark preparation: PASS")

In [ ]:
from pathlib import Path
import json
import numpy as np

root = Path("/content/visionbridge_letter_data")
metadata = json.loads((root / "labels.json").read_text(encoding="utf-8"))

labels = metadata["labels"]
if labels != list("ABCDEFGHIJKLMNOPQRSTUVWXYZ"):
    raise RuntimeError(f"Expected A-Z labels, got {labels}")

print("Labels:", "".join(labels))
print("Split policy:", metadata["split_policy"])

for split in ("train", "val", "test"):
    data = np.load(root / f"{split}.npz")
    x = data["x"]
    y = data["y"]

    if x.ndim != 2 or x.shape[1] != 126:
        raise RuntimeError(f"{split} has invalid shape: {x.shape}")
    if y.ndim != 1 or len(x) != len(y) or len(x) == 0:
        raise RuntimeError(f"{split} has invalid labels")
    if not np.isfinite(x).all():
        raise RuntimeError(f"{split} contains NaN or Inf")

    counts = [int((y == i).sum()) for i in range(26)]
    if any(count == 0 for count in counts):
        raise RuntimeError(f"{split} is missing an A-Z class: {counts}")

    print(f"{split}: samples={len(x)} shape={x.shape}")

print("Prepared dataset contract: PASS")

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/VisionBridge")
PYTHON = Path("/content/visionbridge_train_env/bin/python")
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "backend")

subprocess.run(
    [
        str(PYTHON),
        "-m",
        "app.training.letter_base",
        "--data-dir",
        "/content/visionbridge_letter_data",
        "--output",
        str(REPO / "backend/app/models/weights/letter_base_model.pt"),
        "--epochs",
        "500",
        "--batch-size",
        "128",
        "--lr",
        "0.001",
        "--weight-decay",
        "0.0001",
        "--target-class-accuracy",
        "1.0",
        "--seed",
        "42",
        "--hidden-dim",
        "128",
        "--embedding-dim",
        "64",
        "--dropout",
        "0.10",
    ],
    check=True,
    env=ENV,
)

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/VisionBridge")
PYTHON = Path("/content/visionbridge_train_env/bin/python")
CHECKPOINT = REPO / "backend/app/models/weights/letter_base_model.pt"
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "backend")

if not CHECKPOINT.is_file() or CHECKPOINT.stat().st_size == 0:
    raise RuntimeError("Training did not create a checkpoint")

check = r'''
import sys
from app.models.letter_model import load_checkpoint

model = load_checkpoint(sys.argv[1])
print("CHECKPOINT LOAD: PASS")
print("input_dim =", model.input_dim)
print("hidden_dim =", model.hidden_dim)
print("embedding_dim =", model.embedding_dim)
print("classes =", model.num_classes)
print("labels =", "".join(model.labels))
'''

subprocess.run(
    [str(PYTHON), "-c", check, str(CHECKPOINT)],
    check=True,
    env=ENV,
)

print("Checkpoint bytes:", CHECKPOINT.stat().st_size)
print("Training pipeline: PASS")

## Result

The final checkpoint is:

backend/app/models/weights/letter_base_model.pt

Copy or download that file into the same path in your local VisionBridge repository. The notebook reports validation accuracy after every epoch and evaluates the untouched RealSign test split after training.